# Step 4: Finite-Rank Matrices, Eigenpair Checks, and RHS Bounds

This notebook combines Parts 4A and 4B. Part 4A assembles or loads the finite-rank matrices and verifies the supplied eigenpairs. Part 4B reconstructs the weighted RHS samples and derivative bounds needed by Steps 5, 6, and 7. Both parts read the external `eig.mat`; this notebook does not generate candidate eigenpairs.

Run top to bottom with `notebooks/` as the working directory. The first code cell loads `NS_Numerics` once and defines all output-cache paths. Input files and required fields are checked by the reading functions when used. Interval dtype is required for rigorous enclosures; Float64 mode supplies exploratory approximations.

| Part | Main parameters | Outputs |
|---|---|---|
| 4A: matrices and supplied-eigenpair checks | `R=32`, `N0=18000`, `k=8`, `eta=0.005`, `K6_U=0.449`, `K6_v=0.445` | `integrate_θ_result.mat`, `psi_eig_mat_AB.mat` |
| 4B: weighted RHS and derivative bounds | `R=32`, `12000×6000`, `k=10`, padding 25 then 10 | `rhs.mat`, `rhs_bound.mat` |

The six code cells cover shared initialization, matrix assembly, spectral checks, error diagnostics, RHS reconstruction, and RHS bounds/saving. Assembly parameters and large intermediate arrays remain inside `let` blocks. Part 4A passes its matrices and supplied eigenpairs between the three cells, then clears those arrays after retaining the four scalar results `eps3_U`, `eps4_U`, `eps3_v`, and `eps4_v`. Part 4B passes only `rhs_data` to its final cell; ψ and Q samples leave scope before this handoff, and `rhs_data` is cleared after both saves succeed. The matrix/angular caches and the paired RHS caches keep their independent checks. Cache files must match the current inputs, numeric mode and parameters.

The cached matrices and RHS files must use the same Q, root table, basis ordering, scale, and arithmetic mode.



In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

matrix_path = "../data/generated/psi_eig_mat_AB.mat"
angular_path = "../data/generated/integrate_θ_result.mat"
rhs_path = "../data/generated/rhs.mat"
bound_path = "../data/generated/rhs_bound.mat"



  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


"../data/generated/rhs_bound.mat"

<a id="step-4a"></a>

## Part 4A: Finite-Rank Matrix Assembly and Eigenpair Verification

Construct interval enclosures of the Q and energy bilinear forms on the divergence-free Bessel basis, then verify the supplied finite-rank eigenpairs. These give the paper inputs `eq:45-input-K6`, `eq:45-input-K6v`, and `eq:45-input-errors`.

**Run context.** Use a Julia kernel whose working directory is `notebooks/`, with `dtype = Interval{Float64}` selected in `NS_Numerics.jl` before loading it. The settings here are `N0 = 18000`, `k = 8`, `R = 32`, `eta = 0.005`, `K6_U = 0.449`, and `K6_v = 0.445`. Float64 mode gives approximations without the interval integral certificate.

**Inputs**, relative to the repository root:

| File | Role |
|---|---|
| `data/generated/bessel_zeros.mat` | Degree/root parameters for the poloidal and toroidal basis; roots are refined when interval mode is active |
| `data/external/approx_Q.mat` | Spectral coefficients of the symmetric matrix field Q |
| `data/external/eig.mat` | Supplied eigenvalues and coefficient columns, in the same basis ordering and physical normalization |

**Outputs**, under `data/generated/`:

| File | Stored fields and stage |
|---|---|
| `integrate_θ_result.mat` | `result_even`, `result_odd`: angular tensors on the extended radial grid |
| `psi_eig_mat_AB.mat` | `A_even`, `A_odd`, `B_even`, `B_odd`: matrices after error enlargement and physical scaling |

Check `psi_eig_mat_AB.mat` before any assembly work: if it exists, load the four physical matrices directly. Otherwise load the basis and Q, then load `integrate_θ_result.mat` if present or compute and save its tensors before radial assembly. Enclose and scale newly assembled A/B matrices once, then save them. The caches must match the input fields, basis ordering, and numerical parameters. Part 4B below reconstructs fields from the same external eigenvectors independently of these caches and prepares the RHS data used by Steps 5, 6, and 7. Part 4A does not generate `eig.mat` or the RHS files; Part 4B generates the latter.

**Checks for acceptance.** Each K6 candidate is printed and checked by an active assertion, even before odd. Each epsilon3/epsilon4 upper endpoint is checked against `2.8e-7` inside its parity call, alongside diagonal-positive B, finite residual and Gram-error checks. 

**Reading order:** basis and energy norm → numerical parameters and assembly formulas → cached matrix loading or assembly → deflated spectral certificate → ε₃ and ε₄. Separate code cells assemble/load A/B, read the supplied eigenpairs and run the spectral checks, then calculate ε₃/ε₄ diagnostics. The assembly `let` retains only the physical A/B matrices for the next cells. Matrices and imported eigenpairs are cleared after diagnostics; rerun the matrix-loading and spectral-check cells before repeating those diagnostics. Each calculation block is followed by `GC.gc(true)`. Detailed indexing and proofs are in the [finite-rank assembly note](../docs/technical_notes/finite_rank_assembly.md).

### Basis fields, physical norms, and the matrices being enclosed

Let $B_R=\{x\in\mathbb R^3:|x|<R\}$, with physical radius $\rho=|x|$ and polar angle $\theta$. The finite-dimensional space $V_N$ is spanned by the retained divergence-free basis fields $\Phi_p$. Define

$$
\langle u,v\rangle_E=\int_{B_R}\bigl(\nabla u:\nabla v+\eta\,u\cdot v\bigr)\,dx,
\qquad \|u\|_E^2=\|\nabla u\|_{L^2(B_R)}^2+\eta\|u\|_{L^2(B_R)}^2.
$$

Here the gradient product sums all component derivatives, and `eta = 0.005` is the paper's energy weight $\eta_2$. It is unrelated to the positivity-compensation constant $\eta_1$. The physical matrices are

$$
A_{pq}=\int_{B_R}\Phi_p\cdot Q\Phi_q\,dx,
\qquad B_{pq}=\langle\Phi_p,\Phi_q\rangle_E.
$$

For coefficient vectors $c,d$, their fields satisfy $\langle\sum_pc_p\Phi_p,\sum_pd_p\Phi_p\rangle_E=c^TBd$. A Euclidean coefficient norm therefore does not replace the physical energy norm.

#### One poloidal element contains two channels

With $S_l=P_l(\cos\theta)$ and $H_l=-S_l'$, the returned channels give physical spherical components
\[
\Phi^P=(R^Y S_l,R^XH_l,0),\qquad \Phi^Z=(0,0,R^ZH_l).
\]
Here `Ur_func(...,"X")` returns the negative of the physical poloidal radial coefficient X, so $R^XH_l=XS_l'$. The zero mode is therefore $(\cos\theta,-\sin\theta,0)=e_z$. The toroidal H convention is a whole-mode sign choice. Coefficient rows must match these complete basis elements. Q is the same physical tensor throughout; there is no DQD conversion.

The index $l$ is degree and $j$ is a root within that degree/family. `Y_paras` supplies the coupled Y/X channels of one poloidal element; `Z_paras` supplies one toroidal element. Their radial functions use normalized radius, defined below.

| Field block | Poloidal degrees | Toroidal degrees |
|---|---|---|
| even | $2,4,6,\ldots$ | $1,3,5,\ldots$ |
| odd | $1,3,5,\ldots$ | $2,4,6,\ldots$ |

Within each block, matrix rows pack all poloidal elements first, then toroidal elements, increasing degree and then root. The angular tensors instead use compressed degree index $\lfloor(l+1)/2\rfloor$; root indices are supplied later by the radial factors. See the [basis construction](../docs/technical_notes/bessel_basis.md) and [packing examples](../docs/technical_notes/finite_rank_assembly.md#basis-ordering).

### Coordinates, extended samples, and the angular reduction

Write $s=\rho/R$ for normalized radius. Code variables `r` and `r_ext` contain s; `r_Q` contains Q's stored trigonometric argument $x_Q=(\pi/2)s$. Thus `r = 1/2` represents physical radius 16 and stored Q argument $\pi/4$. Part 4A uses no compactifying β coordinate.

The grid has 18000 subintervals, 18001 physical samples on $s\in[0,1]$, and eight extra samples at each end for angular-tensor derivative bounds: `r_ext` has length 18017. Cropping `9:end-8` aligns each angular tensor with the Bessel samples on `r`. The seven-point Newton–Cotes rule needs an interval count divisible by six, as 18000 is. Extra samples support estimates; they do not enlarge the ball being integrated.

#### Why the half-meridian and parity blocks suffice

For $D_1=\operatorname{diag}(1,-1,1)$, a field in sector $\sigma=\pm1$ satisfies $\Phi_\sigma(\pi-\theta)=\sigma D_1\Phi_\sigma(\theta)$, while $Q(\pi-\theta)=D_1Q(\theta)D_1$. Hence $\Phi_\sigma\cdot Q\Phi_\tau$ has equatorial sign $\sigma\tau$: same-sector integrals double from $[0,\pi/2]$, and cross-sector integrals vanish on $[0,\pi]$. Azimuthal integration supplies $2\pi$. The interval $[0,\pi/2]$ is also the quarter of the angular trigonometric period; the physical meridian itself is $[0,\pi]$.

Native Q entries are even about $x_Q=0$ and odd about $x_Q=\pi/2$. Each scalar angular-contracted entry inherits these radial symmetries and a frequency ceiling `M0*Pi/2` in s. This justifies its reduced-domain Bernstein bounds. The Bessel factors use analytic radial derivative bounds; they are not assumed periodic in s. See the [symmetry proof](../docs/technical_notes/symmetry_and_domains.md#finite-rank-construction).

#### Integrate θ analytically before choosing roots

For components $a,b\in\{1,2,3\}$, set $\Theta_l^1=S_l$ and $\Theta_l^2=\Theta_l^3=H_l$. At each extended radial point,

$$
\mathcal T^{ab}_{ll'}(s)=\int_0^{\pi/2}\Theta_l^a(\theta)Q_{ab}(Rs,\theta)\Theta_{l'}^b(\theta)\sin\theta\,d\theta.
$$

`integrate_θ` evaluates Q's radial coefficients, contracts finite trigonometric angular expansions, and splits by field parity. There is no angular quadrature remainder; interval arithmetic encloses coefficient and rounding uncertainty. Each returned tensor has axes `(radial sample, left compressed degree, right compressed degree, left component, right component)` and shape `(18017,l÷2,l÷2,3,3)`. The implementation assumes even maximum degree and matching coefficient-array sizes for Q's six entries; `M0,M1` are inferred from `A11`. See [angular integration and tensor indices](../docs/technical_notes/finite_rank_assembly.md#angular-integration).

### Radial assembly: values and integral allowances

`eval_Ur` evaluates each channel at the 18001 samples in s. For a fixed degree, columns index roots; Y and X columns share the same poloidal element. `Ur_bound_lst(...,8)` supplies nine rows per degree/root: row $m+1$ bounds the absolute m-th derivative in s, for $m=0,\ldots,8$.

Write $R_p^a(s)$ for component a's radial factor in basis element p. Before physical scaling, a Q matrix entry is

$$
\widehat A_{pq}=\sum_{a,b=1}^3\int_0^1 s^2 R_p^a(s)\mathcal T^{ab}_{l(p),l(q)}(s)R_q^b(s)\,ds.
$$

A poloidal–poloidal pair combines YY, YX, XY, and XX terms; a poloidal–toroidal pair combines YZ and XZ; a toroidal pair has only ZZ. This is why Y and X must not be counted as separate matrix rows.

For each scalar integrand f, `newton_cotes_with_bound` returns a weighted sum and a separate remainder allowance

$$
\left|\int_0^1f(s)\,ds-\operatorname{Quad}(f)\right|
\le\frac{3}{2800}h^8D_8,\qquad h=\frac1{18000},\qquad D_8\ge\|f^{(8)}\|_{L^\infty([0,1])}.
$$

`int_θ_bound` bounds radial derivatives of $\mathcal T$ through order eight using the normalized-radius frequency ceiling. `multinomial_sum` combines these with the analytic Bessel bounds and derivatives of $s^2$ to produce $D_8$. The frequency estimates are independent at each order, so the eight supplied padding layers cover their stencils. The recursive order-eight scheme used elsewhere would require `derivative_padding(8) = 10`; that scheme is not used here. Interval evaluation of a weighted sum alone does not include the integral remainder. See [radial assembly](../docs/technical_notes/finite_rank_assembly.md#radial-assembly) and the [quadrature derivation](../docs/technical_notes/matrix_func.md).

#### B uses orthogonality and the Stokes eigenvalue identity

Let $I_Y=\int_0^1s^2(R^Y_{l,j})^2\,ds$, with analogous definitions for $I_X,I_Z$. Angular orthogonality gives the reduced squared L² norms

$$
\widehat L^P_{l,j}=\frac{I_Y+l(l+1)I_X}{2l+1},
\qquad \widehat L^Z_{l,j}=\frac{l(l+1)I_Z}{2l+1}.
$$

For either family, the corresponding diagonal energy entry is $(\alpha_{l,j}^2/R^2+\eta)\widehat L_{l,j}$, where $\alpha_{l,j}$ is its stored dimensionless spectral parameter. B's off-diagonal zeros follow analytically from basis orthogonality. `get_lap_matrix` applies the same positive factors to the quadrature allowances and packs by field parity. The argument named `β` inside its helpers denotes η here. The factor $R^{-2}$ already converts derivatives to physical radius; no additional derivative rescaling is needed later. See the [energy-matrix derivation](../docs/technical_notes/finite_rank_assembly.md#energy-matrix).

The basis satisfies the natural full-gradient Stokes boundary condition, which gives the Green identity and diagonal energy factor. Raw basis modes are orthogonal but not individually normalized. See [the natural-boundary derivation](../docs/technical_notes/finite_rank_assembly.md#natural-stokes-boundary-conditions); retained normalization is checked separately by the energy Gram matrix.

### Enclose the integrals, then apply physical scaling

In interval mode, `add_error` enlarges each quadrature-value entry by its symmetric integral allowance. Only then are A and B multiplied by

$$
4\pi R^3=\underbrace{2}_{\text{equatorial reflection}}\,
\underbrace{2\pi}_{\text{azimuth}}\,
\underbrace{R^3}_{\rho^2d\rho=R^3s^2ds}.
$$

Thus both the values and their uncertainty receive the physical volume factor. There is no radial doubling: $s\in[0,1]$ already covers the entire ball radius. The matrices saved next enclose the physical bilinear forms defined above, including η in B.

A common scale factor leaves generalized eigenvalues unchanged, but changes energy normalization: the imported V must satisfy $V^TBV\approx I$ for these physically scaled matrices. Matching basis order, roots, R, η, Q, and normalization is part of the input contract. The final Gram diagnostics check its numerical consequences. See [completion of the enclosures](../docs/technical_notes/finite_rank_assembly.md#assembly-verification).

In [2]:
A_even, A_odd, B_even, B_odd = let
    # Common numerical parameters; input loading and large arrays stay in the assembly block.
    N0 = 18000 # radial grid resolution (before extension)
    R = dt(32) # physical truncation radius
    eta = dt("0.005") # energy-norm weight
    k = 8 # derivative order and padding layers for the angular tensors

    if isfile(matrix_path)
        A_even, A_odd, B_even, B_odd = read_matrix(
            matrix_path, ["A_even", "A_odd", "B_even", "B_odd"])
        println("Loaded physical A/B matrices: ", matrix_path)
    else
        # Load parameters defining the divergence-free basis elements.
        # Y_paras parameterizes the poloidal elements (coupled Y/X channels); Z_paras the toroidal
        # elements (Z channel). Both come from the Bessel-zero data.
        Y_paras, Z_paras = load_besselj_zeros("../data/generated/bessel_zeros.mat"; refine=!(dtype==Float64))
        l = max(length(Y_paras), length(Z_paras))

        # Load the operator coefficient/function Q (stored in frequency form).
        # This Q will be used in the bilinear form <φ_i, Q φ_j>.
        Q = load_grad_U("../data/external/approx_Q.mat")

        M0, M1 = size(Q["A11"].freq_func) .* 2 # spectral resolution inferred from Q

        # Extend the radial grid with k ghost points on both sides.
        # This supports finite-difference stencils / derivative bounds used in later steps.
        r_ext = grid(one(dtype), N0, k)
        r = r_ext[k+1:end-k]; # interior grid on [0, 1] (scaled radius)

        theta_block_even, theta_block_odd = let
            if isfile(angular_path)
                theta_block_even, theta_block_odd = read_matrix(
                    angular_path, ["result_even", "result_odd"])
                println("Loaded angular tensors: ", angular_path)
            else
                r_Q = r_ext .* (Pi/dt(2)) # mapped radius used inside Q integration routines

                # Integrate <φ_i, Q φ_j> in the θ direction first.
                # Output tensors retain radial samples and degree/component pairs; angular integration is analytic.
                # integrate_θ returns (field-even, field-odd) — the standard parity-block order.
                theta_block_even, theta_block_odd = integrate_θ(Q, r_Q, l, M1);

                save_matrix(angular_path, Dict(
                    "result_even" => theta_block_even, "result_odd" => theta_block_odd))
                println("Saved angular tensors: ", angular_path)
            end

            (theta_block_even, theta_block_odd)
        end

        # Evaluate radial channels Ur of the divergence-free basis on the interior grid r.
        # Y/X/Z are vector channels, not three independent bases: Ur_Y_lst and Ur_X_lst are the two
        # channels of the SAME poloidal elements (shared roots), and Ur_Z_lst is the toroidal Z channel.
        Ur_Y_lst = eval_Ur(r, Y_paras, "Y")
        Ur_X_lst = eval_Ur(r, Y_paras, "X")
        Ur_Z_lst = eval_Ur(r, Z_paras, "Z")

        # Compute rigorous bounds for Ur and its derivatives up to order k,
        # Derivatives are in normalized radius s; row m+1 bounds order m.
        # These bounds control integral remainders, not the finite-dimensional truncation tail.
        Y_bound_lst, X_bound_lst, Z_bound_lst = Ur_bound_lst(Y_paras, Z_paras, k)

        # Assemble the energy matrix B, including the Laplace eigenvalue and eta weight,
        # with its quadrature error bounds.
        B_even, B_odd, B_even_err, B_odd_err = get_lap_matrix(Ur_Y_lst, Ur_X_lst, Ur_Z_lst, 
            Y_bound_lst, X_bound_lst, Z_bound_lst, Y_paras, Z_paras, r, R, eta)

        # Assemble the operator matrix A (inner products <φ_i, Q φ_j>),
        # using θ-integrated results and rigorous quadrature/error bounds.
        # integrate_θ returns (theta_block_even, theta_block_odd) — already the [field-even, field-odd]
        # order integrate_r expects, so pass it directly (no reordering).
        theta_blocks = [theta_block_even, theta_block_odd]
        A_even, A_odd, A_even_err, A_odd_err = integrate_r(theta_blocks, 
            Ur_Y_lst, Ur_X_lst, Ur_Z_lst, Y_bound_lst, X_bound_lst, Z_bound_lst, r, M0);

        # If we are in interval arithmetic mode (dtype != Float64),
        # incorporate the rigorous error bounds into A and B as interval radii.
        if dtype != Float64
            A_even = add_error(A_even, A_even_err)
            A_odd = add_error(A_odd, A_odd_err)
            B_even = add_error(B_even, B_even_err)
            B_odd = add_error(B_odd, B_odd_err);
        end

        # Rescale to the physical Q and energy Gram matrices on B_R. R^3 converts normalized radial measure; 4π combines azimuthal integration and equatorial symmetry over the whole ball. This does not extend the basis by zero or define a whole-space H1 norm.
        # Applied after add_error, so the interval radii carry the same factor and the
        # enclosures carry the same scaling. Eigenvectors in data/external/eig.mat
        # must use the matching physical normalization; Step 4B reconstructs those vectors.
        A_even .*= dt(4) * Pi * R^3;
        A_odd .*= dt(4) * Pi * R^3;
        B_even .*= dt(4) * Pi * R^3;
        B_odd .*= dt(4) * Pi * R^3;

        save_matrix(matrix_path, Dict(
            "A_even" => A_even, "A_odd" => A_odd,
            "B_even" => B_even, "B_odd" => B_odd))
        println("Saved physical A/B matrices: ", matrix_path)
    end

    (A_even, A_odd, B_even, B_odd)
end
GC.gc(true)


Saved angular tensors: ../data/generated/integrate_θ_result.mat
Integrating r-blocks (Y,X) 1 and 1
Integrating r-blocks (Y,X) 1 and 3
Integrating r-blocks (Y,X) 1 and 5
Integrating r-blocks (Y,X) 1 and 7
Integrating r-blocks (Y,X) 1 and 9
Integrating r-blocks (Y,X) 1 and 11
Integrating r-blocks (Y,X) 1 and 13
Integrating r-blocks (Y,X) 1 and 15
Integrating r-blocks (Y,X) 1 and 17
Integrating r-blocks (Y,X) 1 and 19
Integrating r-blocks (Y,X) 1 and 21
Integrating r-blocks (Y,X) 1 and 23
Integrating r-blocks (Y,X) 1 and 25
Integrating r-blocks (Y,X) 1 and 27
Integrating r-blocks (Y,X) 1 and 29
Integrating r-blocks (Y,X) 1 and 31
Integrating r-blocks (Y,X) 1 and 33
Integrating r-blocks (Y,X) 1 and 35
Integrating r-blocks (Y,X) 1 and 37
Integrating r-blocks (Y,X) 1 and 39
Integrating r-blocks (Y,X) 1 and 41
Integrating r-blocks (Y,X) 1 and 43
Integrating r-blocks (Y,X) 1 and 45
Integrating r-blocks (Y,X) 1 and 47
Integrating r-blocks (Y,X) 1 and 49
Integrating r-blocks (Y,X) 1 and 51
Integ

### Import retained fields and distinguish their indices from basis rows

For each parity block, let V be the imported coefficient matrix and $\Lambda=\operatorname{diag}(\lambda_1,\ldots,\lambda_m)$ the positive retained eigenvalues. Column j represents $\psi_j=\sum_pV_{pj}\Phi_p$. If the block contains $N$ basis elements and retains $m$ fields, then A and B are $N\times N$, V is $N\times m$, and the Gram matrices below are $m\times m$.

$$
G_E=V^TBV,\qquad G_Q=V^TAV,\qquad D=AV\Lambda^{-1}-BV.
$$

Inside `eigen_diagnostics`, `GE`, `GQ`, and `eD` are these quantities; the code variable `Lambda` stores $\Lambda^{-1}$. The approximate eigenproblem is $AV\approx BV\Lambda$. Part 4A imports its candidates and tests them; it does not call an eigensolver.

The following certificates require symmetric physical A and B, positive B and retained eigenvalues, and consistent coefficient coordinates. When the paper uses Q as an inner product, the additional pointwise condition $Q(x)\succeq0$ is required; the finite-dimensional checks here do not prove that condition. The [proof assumptions](../docs/technical_notes/finite_rank_assembly.md#assembly-verification) distinguish this from positivity of a compensated matrix involving $\operatorname{sym}\nabla U$.

### Deflated spectral check

Using $\tau=\texttt{K6\_U}=0.449$ for even and $\tau=\texttt{K6\_v}=0.445$ for odd, `verify_general_eig` forms

$$
S=\tau B-A+(AV)\Lambda^{-1}(AV)^T.
$$

A successful interval LDLᵀ check certifies $S\succ0$. For any nonzero coefficient vector z, this gives

$$
\frac{z^TAz-(V^TAz)^T\Lambda^{-1}(V^TAz)}{z^TBz}<\tau.
$$

When $V^TAz=0$, the deflation term vanishes, proving the required $K_6$ bound on the Q-orthogonal complement. The full-space deflated bound is sufficient for that constrained estimate. It holds for the supplied V without treating the columns as exact eigenvectors. It does not certify their orthonormality or the full-space truncation error; those enter separate estimates.

Each print reports the candidate bound as an interval, rather than a measured eigenvalue. It is immediately followed by a direct assertion using the existing helper, which checks finite, strictly positive LDLᵀ pivots. An uncertified even bound stops execution before the odd check. See the [generalized-eigenvalue certificate](../docs/technical_notes/matrix_func.md#generalized-eigenvalue-certificate--verify_general_eig).

In [3]:
# Coefficient rows must match the basis packing above; columns index retained fields.
# Load precomputed approximate eigenpairs for the generalized eigenproblems:
#   A_even x = λ B_even x,   A_odd x = λ B_odd x.
eig_val_even, eig_vec_even, eig_val_odd, eig_vec_odd = read_matrix("../data/external/eig.mat", 
    ["eig_val_even", "eig_vec_even", "eig_val_odd", "eig_vec_odd"]);

# Certify each candidate bound before starting the next parity block.
K6_U = dt("0.449")
println("K6_U bound: ", K6_U)
@assert verify_general_eig(A_even, B_even, eig_vec_even, eig_val_even[:], K6_U)

K6_v = dt("0.445")
println("K6_v bound: ", K6_v)
@assert verify_general_eig(A_odd, B_odd, eig_vec_odd, eig_val_odd[:], K6_v)
GC.gc(true)



K6_U bound: [0.448999, 0.449]_com
K6_v bound: [0.444999, 0.445]_com


### Gram consistency and energy-weighted residuals

The function `eigen_diagnostics` is called once for each parity block. It uses `sqrt(sum(abs2, M))` to compute the Frobenius norm $\|M\|_F=(\sum_{i,j}|M_{ij}|^2)^{1/2}$, which bounds the spectral norm $\|M\|_2$. Use the upper endpoints of interval results for upper bounds.

#### ε₃: normalized Q Gram error

The code forms $e_Q=I-\Lambda^{-1/2}G_Q\Lambda^{-1/2}$. For $G_Q\succeq0$,

$$
\varepsilon_3=\left\|I-G_Q^{1/2}\Lambda^{-1}G_Q^{1/2}\right\|_2
=\|e_Q\|_2\le\|e_Q\|_F.
$$

The equality follows from the matching singular values of $\Lambda^{-1/2}G_Q^{1/2}$; it does not assume the factors commute. Thus `sqrt(sum(abs2, eQ))` is the printed ε₃ upper bound. A bound below one also certifies positive definite $G_Q$, assuming symmetric $G_Q$ and positive $\Lambda$. The code immediately checks `sup(eps3_bound) ≤ inf(dt("2.8e-7"))` after printing its interval, before forming the independent Gram error or eigen-residual. See the [ε₃ proof](../docs/technical_notes/finite_rank_assembly.md#eps3-bound).

#### ε₄: a separate Gram error and a B-weighted residual

Set $e_E=I-G_E$ and define

$$
\gamma=\|e_E\|_F<1,\qquad
 d=\|B^{-1/2}D\|_F,\qquad
\varepsilon_{4,\mathrm{safe}}=\frac{d}{\sqrt{1-\gamma}}.
$$

The columns $B^{-1}D_j$ represent the weak eigen-residuals divided by $\lambda_j$ in energy coordinates. Since B is diagonal, rowwise division of `eD` by `sqrt.(diag(B))` computes d without a dense matrix square root. The denominator uses the independent Gram error `eps4_gram_error = sqrt(sum(abs2, eE))`, not d and not the Q Gram error.

For $f$ in the span of the retained $\psi_j$ and $u\in V_N$ with $\langle\psi_j,Qu\rangle=0$ for every j, the [ε₄ proof](../docs/technical_notes/finite_rank_assembly.md#eps4-bound) gives

$$
|\langle f,u\rangle_E|\le\varepsilon_{4,\mathrm{safe}}\|f\|_E\|u\|_E.
$$

The assertions require diagonal B with finite positive entries, finite d, and $\gamma<1$ in each parity block. They do not require $d<1$. Once `eps4_bound` is available, the code prints its interval and immediately checks `sup(eps4_bound) ≤ inf(dt("2.8e-7"))`. The Gram bound also proves $G_E\succ0$, since $G_E\succeq(1-\gamma)I$.

Each function call returns `(eps3_bound,eps4_bound)`. The local block returns the four parity bounds used in `eq:45-input-errors`; no separate scalar bounds file is written.

In [4]:
eps3_U, eps4_U, eps3_v, eps4_v = let
    # Compute diagnostic Gram matrices to verify orthogonality and eigenvalue consistency.
    function eigen_diagnostics(A, B, eig_vec, eig_val, dtype, name)
        # Verify the residuals of the eigenvalue problems and the orthogonality.
        GE = eig_vec' * B * eig_vec
        GQ = eig_vec' * A * eig_vec

        # Lambda stores inverse eigenvalues; its elementwise square root is diagonal.
        Lambda = diagm(inv.(eig_val[:]))
        I_mat = Diagonal(ones(dtype, size(GE, 1)))

        # These matrices measure normalized Q consistency, B orthogonality,
        # and the generalized eigen-residual.
        eQ = I_mat - sqrt.(Lambda) * GQ * sqrt.(Lambda)
        eps3_bound = sqrt(sum(abs2, eQ))
        println("eps3_", name, ": ", eps3_bound)
        @assert sup(eps3_bound) <= inf(dt("2.8e-7"))

        # The independent Gram error and generalized eigen-residual enter ε₄.
        # Proofs:
        # ../docs/technical_notes/finite_rank_assembly.md#eps3-bound
        # ../docs/technical_notes/finite_rank_assembly.md#eps4-bound.
        eE = I_mat - GE
        eD = A * eig_vec * Lambda .- B * eig_vec

        # ε₄ uses the B-weighted residual and the independent Gram error.
        # Since B is diagonal,
        #
        #     β_D² = sum_{i,j} |eD[i,j]|² / B[i,i].
        #
        # Thus β_D can be evaluated without constructing B^{-1} or B^{-1/2}.
        @assert isdiag(B)
        @assert all(b -> isfinite(b) && b > zero(dtype), diag(B))

        eps4_residual =
            sqrt(sum(abs2, eD ./ reshape(sqrt.(diag(B)), :, 1)))
        @assert isfinite(eps4_residual)

        # Independently certify ||I - GE||₂ ≤ γ_E < 1 using its Frobenius norm.
        # This gives GE ≥ (1 - γ_E)I and hence
        #
        #     ε₄ ≤ β_D / sqrt(1 - γ_E).
        eps4_gram_error = sqrt(sum(abs2, eE))
        @assert eps4_gram_error < one(dtype)

        eps4_bound =
            eps4_residual / sqrt(one(dtype) - eps4_gram_error)

        println("eps4_", name, ": ", eps4_bound)
        @assert sup(eps4_bound) <= inf(dt("2.8e-7"))
        return eps3_bound, eps4_bound
    end

    eps3_U, eps4_U = eigen_diagnostics(
        A_even, B_even, eig_vec_even, eig_val_even, dtype, "even"
    )

    eps3_v, eps4_v = eigen_diagnostics(
        A_odd, B_odd, eig_vec_odd, eig_val_odd, dtype, "odd"
    )
    eps3_U, eps4_U, eps3_v, eps4_v
end

# Release Part 4A arrays before starting Part 4B.
A_even = A_odd = B_even = B_odd = nothing
eig_val_even = eig_vec_even = eig_val_odd = eig_vec_odd = nothing
GC.gc(true)


eps3_even: [0.0, 4.16785e-8]_com
eps4_even: [0.0, 2.26107e-7]_com
eps3_odd: [0.0, 6.74007e-8]_com
eps4_odd: [0.0, 2.79886e-7]_com


<a id="step-4b"></a>

## Part 4B: Finite-Rank RHS Values and Derivative Bounds

Reconstruct the retained Bessel fields $\psi_i$, form the weighted forcing $wQ\psi_i$, and bound its pure derivatives through order eight for Steps 5, 6 and 7. These supply the residual and finite-rank norm calculations of Steps 5–7. The [RHS note](../docs/technical_notes/finite_rank_rhs.md) gives the longer derivation.

**Run:** use a Julia kernel with working directory `notebooks/` and the project's interval `dtype` selected before loading the module. The `../src` and `../data` paths are relative to this directory. The large padded tensors require substantial memory. The shared first code cell defines the output-cache paths. Part 4B's reconstruction cell checks the pair before loading inputs or allocating grids. When both exist it sets `rhs_data = nothing` and both Part 4B cells skip generation; otherwise the reconstruction cell prepares `rhs_data` and the next cell computes bounds and saves both files. Missing inputs or fields fail in the reading functions. For rigorous enclosures select interval dtype; Float64 runs are exploratory.

| Reads, relative to the repository root | Fields used / producer |
|---|---|
| `data/generated/bessel_zeros.mat` | `Y_zeros`, `Z_zeros`; Step 3, `get_bessel_zeros.ipynb` |
| `data/generated/Q_bound_radial.mat` | `Q_WkInf`; Q-bound export in Step 1, `selfsimilar_profile_verification.ipynb` |
| `data/external/approx_Q.mat` | Six symmetric entries `A11,A12,A13,A22,A23,A33`, with `scl_fac` and `shape` |
| `data/external/eig.mat` | `eig_vec_even`, `eig_vec_odd`; externally supplied generalized-eigenvector coefficients |
| `data/external/UP.mat` | `scl_fac`; the profile's compactification scale only |

**Writes:** `data/generated/rhs.mat` and `data/generated/rhs_bound.mat`, supplying Steps 5, 6 and 7. Both files describe the same inputs, grid, and numeric mode and must be used as a pair.

Part 4A's `integrate_θ_result.mat` and `psi_eig_mat_AB.mat` are not inputs to Part 4B. The external coefficients must nevertheless share its basis ordering, normalization and angular frame. Part 4B does not solve an eigenproblem or compute the inverse fields $\varphi_i$ supplied separately to Steps 5 and 6.

**Expected checks:** prerequisite files and fields, matching coefficient rows and basis, covered enlarged beta interval, and adequate padding. The mode counts come from eig.mat. Part 4B has no final paper-threshold comparison. Its local reconstruction hands only samples, derivative tables and parameters to the final cell through rhs_data; rhs_data is cleared after both saves succeed, with GC after each stage.

### Coordinates and derivative bounds

With $\kappa=\texttt{scl\_fac}$ and Q's support radius $R=32$,

$$
\rho=\kappa\tan\beta,\qquad
r=\rho/R=a\tan\beta,\qquad
a=\kappa/R,\qquad
x_Q=(\pi/2)r,\qquad
\mathrm{Beta}=\arctan(R/\kappa).
$$

The Bessel basis uses $r$; Q interpolation uses $x_Q$. The physical domain is $[0,\mathrm{Beta}]\times[0,\pi/2]$, with ghost samples added for derivative stencils.

Read `Q_WkInf` through order `k=10`, restore its symmetric matrices with `vec_to_mat`, and convert normalized-radius derivative bounds to β bounds once with `r_to_beta`. The priors cover β up to `1.01*Beta`. Q is the physical spherical tensor; `eval_eig_function` reconstructs physical vector components because `Ur_func` compensates the angular H=−S′ factor in its X radial coefficient. No additional frame conjugation is applied.

Refine ψ through order `k` and the weighted RHS through order `k-2`, supplying priors two orders higher. In each mode's output table, `W[m+1,j][c]` bounds derivative order $m$ of component $c$, with `j=1` for β and `j=2` for θ. `beta_bound_chain` collects the β bounds for the product rule.


### Reconstruct both parity blocks and refine ψ bounds

Each column of the imported coefficient matrix V defines $\psi_i=\sum_pV_{pi}\Phi_p$. Its rows index whole basis elements: poloidal elements first, then toroidal elements, ordered by degree and root within each parity. One poloidal coefficient drives both Y and X components; a toroidal coefficient drives Z. The even sector uses even-degree poloidal and odd-degree toroidal elements, with the assignments reversed for the odd sector. See [basis ordering](../docs/technical_notes/finite_rank_assembly.md#basis-ordering).

`eval_eig_function` takes coefficients in **odd, even** order but returns fields in **even, odd** order. Samples keep their signed sums and hence coefficient cancellation. Their axes are `(β sample, θ sample, retained mode, component)`, with components `(Y,X,Z)`.

| Grid or stage | Interior intervals | Padding per side of each axis | Derivative order |
|---|---|---|---|
| ψ samples and refinement | `12000 × 6000` | 25 | Priors through 12; refine through 10 |
| Weighted RHS after cropping | Same | 10 | Priors through 10; refine through 8 |

The counts are intervals: ψ arrays have shape `(12051,6051,J,3)` for J retained modes. Mesh widths remain $h_\beta=\mathrm{Beta}/12000$ and $h_\theta=(\pi/2)/6000$; padding adds samples without changing them. `derivative_padding(10)=15` and `derivative_padding(8)=10`, so ψ needs $15+10=25$ layers. This leaves every exported β derivative through order ten valid on the common ten-layer interval $[-10h_\beta,\mathrm{Beta}+10h_\beta]$ required by the RHS stencils.

`Ur_bound_lst(...,k+2)` supplies radial orders $0{:}12$. `lst_to_mat` attaches the angular magnitude bounds, and `eig_fun_bound` combines them with absolute coefficient values. These triangle-inequality priors discard cancellation. After the tangent chain rule, sampled refinement can sharpen them while its order-eleven and order-twelve priors control the remainder. The analytic envelopes must cover the enlarged normalized-radius interval; see [the Bessel bounds](../docs/technical_notes/bessel_basis.md#polynomial-derivative-bounds).

**Why the angular quarter-domain suffices.** In `(Y,X,Z)` order, the even sector has endpoint parities `(EE,OO,OE)` and the odd sector `(EO,OE,OO)`, where each pair refers to reflection about θ zero and $\pi/2$. Each component therefore has a definite sign under both reflections; differentiation changes signs but preserves absolute-value symmetry. Q's tensor reflection laws preserve these sectors, so the same angular reduction applies to $Q\psi_i$ and $wQ\psi_i$, including angular ghost samples. The angular frequency ceilings are 100 for ψ and 410 for the RHS.

Radially, ψ is odd at the origin in the even sector and even there in the odd sector. This does not supply reflection about `Beta` or radial Bernstein bounds for the tangent/Bessel composition. Explicit radial priors supply that control. See [the symmetry and domain proof](../docs/technical_notes/symmetry_and_domains.md#finite-rank-rhs).

### Form the weighted RHS samples

At each sample and mode, `apply_Q_to_channel_fields` contracts the symmetric matrix with the vector: $(Q\psi_i)_c=\sum_{d=1}^3Q_{cd}\psi_i^d$. It restores off-diagonal entries from the six stored components and inserts no weight. The notebook then forms

$$
w(\beta)=\frac{\tan\beta}{\cos\beta},\qquad
f_i(\beta,\theta)=w(\beta)Q(x_Q(\beta),\theta)\psi_i(r(\beta),\theta).
$$

The radial volume element satisfies $\rho^2\,d\rho=\kappa^3w(\beta)^2\,d\beta$. Thus each factor in later physical L² inner products carries one w. Those consumers separately supply $\sin\theta$ and $4\pi\kappa^3$; the $4\pi$ combines azimuthal integration and same-sector equatorial symmetry. There is no radial doubling or extra $4\pi R^3$ multiplication here. The stored RHS is weighted forcing, not the unweighted physical $Q\psi_i$.

Weight the full 25-layer grid, then crop 15 layers from each end of each coordinate. `rhs.mat` retains ten layers with shape `(12021,6021,J,3)`. Beyond `Beta`, these are the analytic extensions needed by the stencils; they are not the physical zero extension of the compactly supported forcing. The consumers retain them for refinement and remove them before quadrature. A nested `let` returns only these RHS samples and the small refined ψ bounds. The reconstructed ψ and Q samples leave scope before `GC.gc(true)` and before either MAT file is saved.

In [5]:
# Reset the reconstruction handoff before allocating arrays.
rhs_data = nothing
GC.gc(true)
rhs_data = let
    # k=10 supplies product priors two orders above the final W8Inf refinement.
    R = dt(32)
    k = 10
    padding_rhs = derivative_padding(k-2)
    padding_psi = derivative_padding(k) + padding_rhs
    N0_glb, N1_glb = 12000, 6000
    L1 = Pi / dt(2)

    if isfile(rhs_path) && isfile(bound_path)
        println("RHS caches found; skipping")
        nothing
    else
        scl_fac = dt(read_matrix("../data/external/UP.mat", ["scl_fac"]))
        a = scl_fac / R
        Beta = atan(R / scl_fac)
        beta_prior_max = Beta * dt("1.01")

        # Q's file contains derivatives in normalized radius, not in β.
        # Read orders 0:k and transform exactly once.
        Q_radial_table = read_matrix("../data/generated/Q_bound_radial.mat", ["Q_WkInf"])
        Q_radial_bounds = vec_to_mat(Q_radial_table[1:k+1, :])
        Q_beta_bounds = r_to_beta(Q_radial_bounds, a; β_max=beta_prior_max)

        Y_paras, Z_paras = load_besselj_zeros("../data/generated/bessel_zeros.mat"; refine=!(dtype == Float64))
        # Load Q coefficients in their stored tensor frame; this does not differentiate UP.
        Q = load_grad_U("../data/external/approx_Q.mat")
        eig_vec_even, eig_vec_odd = read_matrix("../data/external/eig.mat", ["eig_vec_even", "eig_vec_odd"])

        # Retain only RHS samples and small ψ bounds after this stage.
        rhs_even, rhs_odd, psi_norm_lst_even, psi_norm_lst_odd = let
            # N0_glb and N1_glb count physical intervals; padding adds endpoint stencil samples.
            β_glb = grid(Beta, N0_glb, padding_psi)
            θ_glb = grid(L1, N1_glb, padding_psi)
            @assert beta_prior_max < Pi / dt(2)
            @assert maximum(abs.(β_glb)) <= beta_prior_max
            r_glb = tan.(β_glb) * a

            eig_psi_even, eig_psi_odd = eval_eig_function(
                eig_vec_odd, eig_vec_even, Y_paras, Z_paras, r_glb, θ_glb)
            # Basis bounds include angular magnitude factors before multiplying by |eig_vec|.
            U_bound_even, U_bound_odd = lst_to_mat(Ur_bound_lst(Y_paras, Z_paras, k+2)...)
            eig_bound_even_prior = r_to_beta(eig_fun_bound(U_bound_even, eig_vec_even), a; β_max=beta_prior_max)
            eig_bound_odd_prior = r_to_beta(eig_fun_bound(U_bound_odd, eig_vec_odd), a; β_max=beta_prior_max)

            # Refine through order 10 while retaining a common ten-layer beta target interval.
            psi_norm_lst_even = refine_channel_bounds(eig_psi_even, eig_bound_even_prior,
                scl_fac, [Beta, L1], k; angular_frequency=dt(100), padding=padding_psi, label="ψ even")
            psi_norm_lst_odd = refine_channel_bounds(eig_psi_odd, eig_bound_odd_prior,
                scl_fac, [Beta, L1], k; angular_frequency=dt(100), padding=padding_psi, label="ψ odd")

            # The Q interpolation coordinate is (pi/2) times the normalized Bessel radius.
            Q_pts = interpolate(Q, r_glb .* (Pi / dt(2)), θ_glb)
            rhs_even, rhs_odd = apply_Q_to_channel_fields(Q_pts, [eig_psi_even, eig_psi_odd])
            # Each stored RHS carries one radial L2 weight; angular/measure factors come later.
            weight_glb = tan.(β_glb) ./ cos.(β_glb)
            rhs_even .*= weight_glb
            rhs_odd .*= weight_glb
            # Retain the ten-layer RHS stencil grid after weighting the full ψ grid.
            crop_rhs = padding_psi - padding_rhs
            rhs_even = @view rhs_even[crop_rhs+1:end-crop_rhs, crop_rhs+1:end-crop_rhs, :, :]
            rhs_odd = @view rhs_odd[crop_rhs+1:end-crop_rhs, crop_rhs+1:end-crop_rhs, :, :]

            (rhs_even, rhs_odd, psi_norm_lst_even, psi_norm_lst_odd)
        end
        # ψ and Q sample arrays are now out of scope, before any MAT save.
        GC.gc(true)

        # Only these results are needed by the next cell.
        (; rhs_even, rhs_odd, psi_norm_lst_even, psi_norm_lst_odd,
           Q_beta_bounds, scl_fac, Beta, beta_prior_max, k, padding_rhs, L1)
    end
end
GC.gc(true)


ψ YX 1/96
ψ YX 2/96
ψ YX 3/96
ψ YX 4/96
ψ YX 5/96
ψ YX 6/96
ψ YX 7/96
ψ YX 8/96
ψ YX 9/96
ψ YX 10/96
ψ YX 11/96
ψ YX 12/96
ψ YX 13/96
ψ YX 14/96
ψ YX 15/96
ψ YX 16/96
ψ YX 17/96
ψ YX 18/96
ψ YX 19/96
ψ YX 20/96
ψ YX 21/96
ψ YX 22/96
ψ YX 23/96
ψ YX 24/96
ψ YX 25/96
ψ YX 26/96
ψ YX 27/96
ψ YX 28/96
ψ YX 29/96
ψ YX 30/96
ψ YX 31/96
ψ YX 32/96
ψ YX 33/96
ψ YX 34/96
ψ YX 35/96
ψ YX 36/96
ψ YX 37/96
ψ YX 38/96
ψ YX 39/96
ψ YX 40/96
ψ YX 41/96
ψ YX 42/96
ψ YX 43/96
ψ YX 44/96
ψ YX 45/96
ψ YX 46/96
ψ YX 47/96
ψ YX 48/96
ψ YX 49/96
ψ YX 50/96
ψ YX 51/96
ψ YX 52/96
ψ YX 53/96
ψ YX 54/96
ψ YX 55/96
ψ YX 56/96
ψ YX 57/96
ψ YX 58/96
ψ YX 59/96
ψ YX 60/96
ψ YX 61/96
ψ YX 62/96
ψ YX 63/96
ψ YX 64/96
ψ YX 65/96
ψ YX 66/96
ψ YX 67/96
ψ YX 68/96
ψ YX 69/96
ψ YX 70/96
ψ YX 71/96
ψ YX 72/96
ψ YX 73/96
ψ YX 74/96
ψ YX 75/96
ψ YX 76/96
ψ YX 77/96
ψ YX 78/96
ψ YX 79/96
ψ YX 80/96
ψ YX 81/96
ψ YX 82/96
ψ YX 83/96
ψ YX 84/96
ψ YX 85/96
ψ YX 86/96
ψ YX 87/96
ψ YX 88/96
ψ YX 89/96
ψ YX 90/96
ψ YX 91/96
ψ YX 92/

### Combine product priors and refine RHS bounds

`beta_bound_chain` supplies the refined ψ derivatives through order ten. `B` bounds derivatives of w, and `Q_beta_bounds` bounds derivatives of the composed Q. For component c of retained mode i, the product-rule estimate is

$$
\|\partial_\beta^m f_i^c\|_\infty
\le\sum_{p+q+s=m}\frac{m!}{p!q!s!}
\|w^{(p)}\|_\infty
\sum_{d=1}^3\|\partial_\beta^q Q_{cd}\|_\infty
\|\partial_\beta^s\psi_i^d\|_\infty.
$$

All factors cover the common ten-layer radial stencil interval. The Q priors come from the labelled radial table converted above, rather than interpreting a β table as radial data. `multinomial_sum(...; last_flag=false)` returns every order $0{:}10$; its ordered scalar–matrix–vector product performs the component contraction.

The final `W8Inf` calls use order-nine and order-ten product priors to refine through order eight. They receive the already cropped RHS with `padding=10` and the helper's default `crop=0`: ten layers are exactly `derivative_padding(8)`. The angular ceiling 410 supplies the deferred θ estimates justified by the sector symmetries above. The resulting tables bound pure β and θ derivatives on the physical rectangle $[0,\mathrm{Beta}]\times[0,\pi/2]$ for the later residual and quadrature calculations; only the saved sample tensors retain the ten-layer halo. See [the product and padding derivation](../docs/technical_notes/finite_rank_rhs.md).

### Save derivative bounds and hand off to Steps 5, 6 and 7

The reconstruction cell above checks `data/generated/rhs.mat` and `data/generated/rhs_bound.mat` together. If both exist, `rhs_data` is `nothing` and the cell below also skips calculation and saving; Steps 5, 6 and 7 read the caches directly. If either is absent, the first cell prepares weighted samples and ψ bounds, and the cell below computes the product priors and RHS derivative bounds. Saving begins only after the samples and derivative bounds have both been calculated. Both files are rebuilt from the same inputs.

| Logical fields | Array layout |
|---|---|
| `rhs_even`, `rhs_odd` | `(12021,6021,J,3)`: β, θ, retained mode, component |
| `rhs_bound_even`, `rhs_bound_odd` | `(3,9,2,J)`: component, order $0{:}8$, direction `(β,θ)`, mode |
| `rhs_prior_even`, `rhs_prior_odd` | `(3,J,11,1)`: component, mode, β order $0{:}10$, singleton block |

The prior's final axis is a singleton block, not a derivative direction. In interval mode, `save_matrix` writes `_lo`/`_hi` endpoint fields, which the consumers reconstruct with `read_matrix` using the logical names above. J is the column count of the corresponding even or odd coefficient matrix; it is not hard-coded.

Step 5 reads the even samples, refined bounds and priors; Step 6 reads their odd counterparts. Their imported $\varphi_i$ are approximate inverse solutions driven by $Q\psi_i$, distinct from the Bessel combinations ψ constructed here. The reconstruction `let` releases ψ, Q and grid samples before returning `rhs_data`. The final `let` uses its weighted samples, derivative tables and parameters to calculate and save both files. After successful saving, `rhs_data` is cleared; on failure it remains available for retrying the final cell. `GC.gc(true)` follows both stages. See the [data dictionary](../docs/data.md) for the file contract.

Step 4B stores g_j=wQpsi_j without division by the retained generalized eigenvalue nu_j. Steps 5–6 use these samples unchanged in residual assembly, while their forcing Gram divides by nu_i nu_j; the odd cross-Grams divide by the row eigenvalue once. Step 7 (`verify_auxiliary_field_bound.ipynb`) also reads the odd samples/bounds and combines them with fixed centers `c_j^0/nu_j`; a separate Step 6 residual-Gram allowance controls the source-coefficient error. See [normalization at the handoff](../docs/technical_notes/finite_rank_rhs.md#71-normalization-at-the-handoff-to-inverse-verification) for the physical measure, signs and matched-input requirements.

In [6]:
if rhs_data !== nothing
    let
        (; rhs_even, rhs_odd, psi_norm_lst_even, psi_norm_lst_odd,
           Q_beta_bounds, scl_fac, Beta, beta_prior_max, k, padding_rhs, L1) = rhs_data

        eig_bound_even = beta_bound_chain(psi_norm_lst_even)
        eig_bound_odd = beta_bound_chain(psi_norm_lst_odd)
        # Keep the product ordered: scalar weight bounds, matrix Q bounds, vector psi bounds.
        B = tan_over_cos_deriv_Linf_bounds(beta_prior_max, k)
        rhs_β_norms_even_prior = multinomial_sum(k, B, Q_beta_bounds, eig_bound_even; last_flag=false)
        rhs_β_norms_odd_prior = multinomial_sum(k, B, Q_beta_bounds, eig_bound_odd; last_flag=false)

        # These arrays already have ten layers: refine at order 8 without another crop.
        rhs_norm_lst_even = refine_channel_bounds(rhs_even, rhs_β_norms_even_prior,
            scl_fac, [Beta, L1], k-2; angular_frequency=dt(410), padding=padding_rhs, label="RHS even")
        rhs_norm_lst_odd = refine_channel_bounds(rhs_odd, rhs_β_norms_odd_prior,
            scl_fac, [Beta, L1], k-2; angular_frequency=dt(410), padding=padding_rhs, label="RHS odd")

        # Save the matched pair only after both calculations have succeeded.
        save_matrix(rhs_path, Dict("rhs_even" => rhs_even, "rhs_odd" => rhs_odd))
        # Preserve the exact array layouts expected by Steps 5 and 6:
        # refined: component × derivative order × direction × mode
        # prior:   component × mode × derivative order × singleton block
        save_matrix(bound_path, Dict(
            "rhs_bound_even" => stack(stack(rhs_norm_lst_even)),
            "rhs_prior_even" => stack(rhs_β_norms_even_prior),
            "rhs_bound_odd" => stack(stack(rhs_norm_lst_odd)),
            "rhs_prior_odd" => stack(rhs_β_norms_odd_prior)))
        println("Saved RHS and bounds")
        nothing
    end
    # Clear the cross-cell samples only after both saves succeed.
    rhs_data = nothing
end
GC.gc(true)


RHS even 1 / 18
RHS even 2 / 18
RHS even 3 / 18
RHS even 4 / 18
RHS even 5 / 18
RHS even 6 / 18
RHS even 7 / 18
RHS even 8 / 18
RHS even 9 / 18
RHS even 10 / 18
RHS even 11 / 18
RHS even 12 / 18
RHS even 13 / 18
RHS even 14 / 18
RHS even 15 / 18
RHS even 16 / 18
RHS even 17 / 18
RHS even 18 / 18
RHS odd 1 / 25
RHS odd 2 / 25
RHS odd 3 / 25
RHS odd 4 / 25
RHS odd 5 / 25
RHS odd 6 / 25
RHS odd 7 / 25
RHS odd 8 / 25
RHS odd 9 / 25
RHS odd 10 / 25
RHS odd 11 / 25
RHS odd 12 / 25
RHS odd 13 / 25
RHS odd 14 / 25
RHS odd 15 / 25
RHS odd 16 / 25
RHS odd 17 / 25
RHS odd 18 / 25
RHS odd 19 / 25
RHS odd 20 / 25
RHS odd 21 / 25
RHS odd 22 / 25
RHS odd 23 / 25
RHS odd 24 / 25
RHS odd 25 / 25
Saved RHS and bounds
